Daniel Yu & Jordan Wang

Spring 2026

CS 443: Bio-inspired Machine Learning

# Extension 3: CS 343 Softmax Network as Linear Decoder

## Hypothesis

We wanted to investigate how the Adam optimizer and validation set support improve our CS 343 softmax network when used as a linear decoder for Hebbian activations.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd

import sys
sys.path.append('../Project 1 - Hebbian Learning')
from image_datasets import get_dataset, train_val_split, preprocess_nonlinear
from decoder_nets import NonlinearDecoder
from hebb_net import HebbNet

plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.rcParams.update({'font.size': 14})
np.set_printoptions(suppress=True, precision=3)

%load_ext autoreload
%autoreload 2

## Setup

First, we need to import the softmax_layer.py and optimizer.py from CS 343 projects, then modify softmax_layer.py to support Adam optimizer and validation sets.

In [ ]:
# Load MNIST dataset
x_train_mnist, y_train_mnist, x_test_mnist, y_test_mnist = get_dataset('mnist', verbose=True, norm_method='center')
x_train_mnist_split, y_train_mnist_split, x_val_mnist, y_val_mnist = train_val_split(x_train_mnist, y_train_mnist)

print(f'Train: {x_train_mnist_split.shape}, Val: {x_val_mnist.shape}, Test: {x_test_mnist.shape}')

# Load pre-trained Hebbian network
hebb_mnist = HebbNet(784, 2000, k=6, inhib_value=-0.4, load_wts=True, saved_wts_path='../Project 1 - Hebbian Learning/export/wts_centered.npy')

# Compute Hebbian activations for train, val, and test sets
x_train_hebb = tf.concat([hebb_mnist.net_in(x_train_mnist_split[i:i+1000]) for i in range(0, len(x_train_mnist_split), 1000)], axis=0)
x_val_hebb = tf.concat([hebb_mnist.net_in(x_val_mnist[i:i+1000]) for i in range(0, len(x_val_mnist), 1000)], axis=0)
x_test_hebb = tf.concat([hebb_mnist.net_in(x_test_mnist[i:i+1000]) for i in range(0, len(x_test_mnist), 1000)], axis=0)

print(f'Hebbian train feats: {x_train_hebb.shape}')
print(f'Hebbian val feats: {x_val_hebb.shape}')
print(f'Hebbian test feats: {x_test_hebb.shape}')

Dataset: mnist
x_train: (60000, 784) <dtype: 'float32'>
y_train: (60000,) <dtype: 'uint8'>
x_test:  (10000, 784) <dtype: 'float32'>
y_test:  (10000,) <dtype: 'uint8'>
Train: (54000, 784), Val: (6000, 784), Test: (10000, 784)
Loaded stored wts.
Hebbian train feats: (54000, 2000)
Hebbian val feats: (6000, 2000)
Hebbian test feats: (10000, 2000)
Hebbian train feats: (54000, 2000)
Hebbian val feats: (6000, 2000)
Hebbian test feats: (10000, 2000)


## Experiment 1: CS 343 Softmax with Adam Optimizer

In this experiment, we will compare the CS 343 softmax network with Adam optimizer to TensorFlow's LinearDecoder on Hebbian activations. Analyze training speed, convergence behavior, and final accuracy.

**Softmax network modifications needed:**
- Switch from SGD to Adam optimizer (two Adam objects: one for weights, one for bias)
- Add validation set support (x_val, y_val parameters)
- Convert print outputs to epochs instead of iterations
- Return both train and validation loss lists

In [ ]:
from decoder_nets import LinearDecoder
from softmax_layer import SoftmaxLayer

# Reproducibility
tf.random.set_seed(0)
np.random.seed(0)

H = x_train_hebb.shape[1]
C = 10

# 1) TensorFlow LinearDecoder baseline
linear_dec_tf = LinearDecoder(input_feats_shape=(H,), C=C)
linear_dec_tf.compile(loss='cross_entropy', lr=1e-3)

start_time = time.time()
train_loss_tf, val_loss_tf, val_acc_tf, num_epochs_tf = linear_dec_tf.fit(
    x_train_hebb, y_train_mnist_split,
    x_val=x_val_hebb, y_val=y_val_mnist,
    batch_size=256,
    max_epochs=200,
    patience=5,
    lr_patience=3,
    lr_max_decays=4,
    val_every=1,
    print_every=5,
    verbose=False
)
time_tf = time.time() - start_time

test_acc_tf, _ = linear_dec_tf.evaluate(x_test_hebb, y_test_mnist)

# 2) CS 343 Softmax with Adam + validation support
softmax_cs343 = SoftmaxLayer(num_output_units=C)

start_time = time.time()
train_loss_cs343, val_loss_cs343 = softmax_cs343.fit(
    x_train_hebb.numpy(),
    y_train_mnist_split.numpy(),
    n_epochs=num_epochs_tf,
    lr=1e-3,
    mini_batch_sz=256,
    reg=0.0,
    r_seed=0,
    verbose=0,
    x_val=x_val_hebb.numpy(),
    y_val=y_val_mnist.numpy(),
    val_freq=1
)
time_cs343 = time.time() - start_time

y_pred_cs343 = softmax_cs343.predict(x_test_hebb.numpy())
test_acc_cs343 = np.mean(y_pred_cs343 == y_test_mnist.numpy())

y_val_pred_cs343 = softmax_cs343.predict(x_val_hebb.numpy())
val_acc_cs343 = np.mean(y_val_pred_cs343 == y_val_mnist.numpy())

print(f'TensorFlow LinearDecoder: {100*float(test_acc_tf.numpy()):.2f}% test, {100*max(val_acc_tf):.2f}% val, {time_tf:.1f}s')
print(f'CS343 Softmax (Adam):    {100*test_acc_cs343:.2f}% test, {100*val_acc_cs343:.2f}% val, {time_cs343:.1f}s')
print(f'Epochs used for both runs: {num_epochs_tf}')

---------------------------------------------------------------------------
Dense layer output(output) shape: [1, 10]
---------------------------------------------------------------------------
Learning rate before decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0010000000474974513>
Learning rate after decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0005000000237487257>
Learning rate before decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0010000000474974513>
Learning rate after decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0005000000237487257>
Learning rate before decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0005000000237487257>
Learning rate after decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0002500000118743628>
Learning rate before decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.00050000002

## Results & Analysis

We compared both models on the same Hebbian MNIST features and tracked test accuracy, validation accuracy, runtime, and training epochs.

- TensorFlow LinearDecoder: 91.86% test, 93.15% validation, 25.4s
- CS343 Softmax (Adam): 91.28% test, 92.32% validation, 25.1s
- Epochs used for both runs: 30

First, the CS343 softmax with Adam performed close to the TensorFlow baseline. The test accuracy difference was 0.58 points, so the custom model stayed in the same range.

Next, runtime was also very close in this run. CS343 finished slightly faster, but the gap was small.

Then, looking at the bigger picture, the behavior matches our hypothesis. Adding Adam and validation support made the CS343 implementation stable and competitive.

Finally, this extension now includes all required updates in softmax_layer.py: Adam for weights and bias, validation arguments in fit(), epoch-based logging with val_freq, and returning both train and validation loss histories.

## Conclusion

Overall, this extension did what we wanted. After adding Adam and validation support, the CS343 softmax trained similarly to the TensorFlow linear decoder on the same Hebbian features. The accuracy was close, and runtime was also close, so the custom implementation is both valid and practical for this task.